In [ ]:
from utils import embed_lecture_slides, create_real_estate_dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure pandas display for better readability
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 1000)
pd.set_option('display.precision', 2)

# Set plot style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

In [ ]:
embed_lecture_slides('03_NeuralNetworks/03-regression-deck.html#/machine-learning-pipeline-1/4')

# Introduction to Pandas - Feature Analysis and Data Processing

This notebook will guide you through the essential pandas skills needed to prepare real-world data for machine learning. By the end, you'll understand how to inspect, clean, and analyze datasets like the housing data you'll work with in the exercise.

## What We'll Cover

1. **Why Pandas?** — Motivation and comparison with NumPy
2. **Inspecting a New Dataset** — First steps when encountering unfamiliar data
3. **Basic Handling of DataFrames** — Core data structures and operations
4. **Data Manipulation and Cleaning** — Dealing with missing values, outliers, and inconsistencies
5. **Understanding Feature Relationships** — Correlation, visualization, and feature intuition
6. **Connecting Pandas to Modeling** — Bridging to scikit-learn and the ML pipeline

---

## 1. Why Pandas? (Pandas vs NumPy)

When working with real-world datasets, you'll quickly encounter challenges that NumPy alone cannot elegantly solve. Let's understand **why** pandas is essential for data science.

### The Challenge: Real-World Data is Heterogeneous

Imagine a real estate dataset with:
- **Numerical data:** rent prices, square meters, number of rooms
- **Categorical data:** city names, heating types, apartment condition
- **Text data:** descriptions, addresses
- **Missing values:** some apartments don't have balcony info, garden data, etc.
- **Time data:** year built, availability dates

NumPy arrays require **homogeneous data** (all elements must be the same type). This makes them fast but inflexible for real-world tabular data.

**Pandas** solves this by providing:
- **Series** (1D labeled array) — like a column in a spreadsheet
- **DataFrame** (2D labeled table) — like an entire spreadsheet with rows and columns

### Example: NumPy vs Pandas

Let's compare how NumPy and Pandas handle mixed-type data:

In [ ]:
# NumPy approach - forces everything to strings!
numpy_data = np.array([
    ['Berlin', 850.5, 45.0, 2],
    ['Munich', 1200.0, 60.0, 3],
    ['Hamburg', 950.0, 52.0, 2]
])

print("NumPy Array (all becomes strings):")
print(numpy_data)
print(f"Data type: {numpy_data.dtype}")
print('Can we calculate the mean rent?')
try:
    print(numpy_data[:, 1].mean())
except Exception as e:
    print('No - strings!')

print("\n" + "="*60 + "\n")

# Pandas approach - preserves types!
pandas_data = pd.DataFrame({
    'city': ['Berlin', 'Munich', 'Hamburg'],
    'rent': [850.5, 1200.0, 950.0],
    'sqm': [45.0, 60.0, 52.0],
    'rooms': [2, 3, 2]
})

print("Pandas DataFrame (types preserved):")
print(pandas_data)
print(f"\nData types:")
print(pandas_data.dtypes)
print(f"\nMean rent: {pandas_data['rent'].mean():.2f} €")

### Key Advantages of Pandas

1. **Named columns** — Access data by meaningful names, not just indices  
2. **Mixed data types** — Numerical, categorical, text in one table  
3. **Missing value handling** — Built-in `NaN` support  
4. **Rich operations** — Filtering, grouping, merging, joining  
5. **Integration** — Works seamlessly with scikit-learn, matplotlib, seaborn  
6. **Expressive** — Write less code, more readable  

**Bottom line:** For tabular data analysis and machine learning preprocessing, pandas is indispensable. 

---

## 2. Inspecting a New Dataset

Let's load a real dataset and learn how to **audit** it before modeling.

### Loading Data

Pandas can read data from many sources:
- **CSV files:** `pd.read_csv('file.csv')`
- **Excel:** `pd.read_excel('file.xlsx')`
- **JSON:** `pd.read_json('file.json')`
- **SQL databases:** `pd.read_sql(query, connection)`
- **Web:** `pd.read_html(url)` or `pd.read_csv(url)`

For this exercise, you'll work with CSV data from real estate listings. The one used in this exercise was crafted for demonstration purposes.

In [ ]:
df = create_real_estate_dataset(n_samples=500)

### First Look: `.head()` and `.tail()`

Always start by looking at the first few rows to get a sense of the data structure.

In [ ]:
# View first 5 rows
df.head()

In [ ]:
# View last 3 rows
df.tail(3)

### Shape and Structure

**Critical first questions:**
- How many rows (samples)?
- How many columns (features)?
- What are the data types?
- Are there missing values?

In [ ]:
# Shape: (rows, columns)
print(f"Dataset shape: {df.shape}")
print(f"  -> {df.shape[0]} apartments (rows)")
print(f"  -> {df.shape[1]} features (columns)")

print("\n" + "="*60)

# Column names
print(f"\nColumn names:")
print(df.columns.tolist())

print("\n" + "="*60)

# Quick overview with .info()
print("\nDetailed info:")
df.info()

**Key observations from `.info()`:**
- `Non-Null Count` shows how many non-missing values exist per column
- `Dtype` shows the data type (float64, int64, bool, object for strings)
- Memory usage helps understand dataset size

### Summary Statistics

The `.describe()` method provides statistical summaries:

In [ ]:
# Statistical summary for numerical columns
df.describe(include=['float', 'int'])

**Understanding the statistics:**
- **count:** Number of non-missing values
- **mean:** Average value
- **std:** Standard deviation (spread/variability)
- **min/max:** Range
- **25%, 50%, 75%:** Quartiles (percentiles)

**Red flags to watch for:**
- Large differences between mean and median → skewed distribution or outliers
- Unrealistic min/max values (e.g., negative rent, 1000 rooms)
- High standard deviation relative to mean → high variability or quality issues

In [ ]:
# Summary for categorical (object) columns
df.describe(include='object')

In [ ]:
# Check unique values for categorical columns
for col in ['city', 'heatingType']:
    print(df[col].value_counts())
    print(f"  → {df[col].nunique()} unique values")

### Checking for Missing Values

Missing data is common in real-world datasets. Let's identify where values are missing:

In [ ]:
# Count missing values per column
missing_counts = df.isnull().sum()
missing_percent = (missing_counts / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing_counts,
    'Percentage': missing_percent
}).sort_values('Missing Count', ascending=False)

print("Missing Values Summary:")
print(missing_df[missing_df['Missing Count'] > 0])

### Visualizing the Target Variable

Let's visualize the distribution of our target variable `totalRent`:

In [ ]:
# Histogram of totalRent
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
df['totalRent'].hist(bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Total Rent (€)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.title('Distribution of Total Rent', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
df.boxplot(column='totalRent', vert=False)
plt.xlabel('Total Rent (€)', fontsize=12)
plt.title('Box Plot: Total Rent', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Visualizing Feature Relationships

Let's examine how each numerical feature relates to our target variable `totalRent` through scatter plots:

In [ ]:
# Select numerical features for visualization
numerical_features = ['baseRent', 'serviceCharge', 'livingSpace', 'noRooms', 'yearBuilt', 'distanceFromCenter']
target = 'totalRent'

# Create a grid of scatter plots
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, feature in enumerate(numerical_features):
    ax = axes[i]
    
    # Create scatter plot
    ax.scatter(df[feature], df[target], alpha=0.5, s=20, edgecolor='k', linewidth=0.5)
    ax.set_xlabel(feature, fontsize=11, fontweight='bold')
    ax.set_ylabel('Total Rent (€)', fontsize=11, fontweight='bold')
    ax.set_title(f'{feature} vs Total Rent', fontsize=12, fontweight='bold')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---

## 3. Basic Handling of DataFrames

Now that we understand what's in our data, let's learn how to manipulate it effectively.

### Core Data Structures Revisited

**Series:** A 1D labeled array (think: one column)

In [ ]:
# Extract a single column as a Series
rent_series = df['totalRent']

print(f"Type: {type(rent_series)}")
print(f"Shape: {rent_series.shape}")
print(f"\nFirst 5 values:")
print(rent_series.head())

**DataFrame:** A 2D table of Series with a shared index


In [ ]:
# Extract several column as a DataFrame
data = df[['city', 'totalRent', 'livingSpace']]

print(f"Type: {type(data)}")
print(f"Shape: {data.shape}")
print(f"\nFirst 5 values:")
print(data.head())

### Accessing Data

There are several ways to access data in a DataFrame:

In [ ]:
# 1. Column access by name
print("1. Single column:")
print(df['city'].head(3))

print("\n2. Multiple columns:")
print(df[['city', 'totalRent', 'livingSpace']].head(3))

print("\n3. Using .loc[] for label-based indexing:")
# .loc[rows, columns]
print(df.loc[0:4, ['city', 'totalRent']])  # First 5 rows, specific columns

print("\n4. Using .iloc[] for position-based indexing:")
# .iloc[row_positions, column_positions]
print(df.iloc[0:3, 0:3])  # First 3 rows, first 3 columns

### Filtering and Subsetting

Boolean indexing allows you to filter rows based on conditions:

In [ ]:
# Filter: Find apartments in Berlin only
berlin_apts = df[df['city'] == 'Berlin']
print(f"Berlin apartments: {len(berlin_apts)}")
print(berlin_apts.head(3))

print("\n" + "="*60 + "\n")

# Multiple conditions: Berlin AND > 60 sqm
large_berlin = df[(df['city'] == 'Berlin') & (df['livingSpace'] > 60)]
print(f"Large Berlin apartments: {len(large_berlin)}")

print("\n" + "="*60 + "\n")

# OR conditions: Berlin OR Munich
berlin_munich = df[(df['city'] == 'Berlin') | (df['city'] == 'Munich')]
print(f"Berlin or Munich: {len(berlin_munich)}")

# Alternative: using .isin()
berlin_munich_alt = df[df['city'].isin(['Berlin', 'Munich'])]
print(f"Using .isin(): {len(berlin_munich_alt)}")

### Sorting

Sort data by one or more columns:

In [ ]:
# Sort by totalRent (descending) - most expensive first
df_sorted = df.sort_values('totalRent', ascending=False)
print("Most expensive apartments:")
print(df_sorted[['city', 'totalRent', 'livingSpace', 'noRooms']].head())

# Sort by multiple columns
df_multi_sort = df.sort_values(['city', 'totalRent'], ascending=[True, False])
print("\nSorted by city (A-Z), then by rent (high to low) within each city:")
print(df_multi_sort[['city', 'totalRent']].head(10))

### Grouping Operations

**Grouping** is one of pandas' features for aggregating data by categories.

**The "split-apply-combine" paradigm:**
1. **Split:** Divide data into groups based on some criterion
2. **Apply:** Compute a function (mean, sum, count, etc.) for each group
3. **Combine:** Merge results into a single data structure

This is essential for understanding patterns in categorical data (e.g., average rent by city).

In [ ]:
# Basic grouping: Average rent by city
city_groups = df.groupby('city')['totalRent'].mean()

print("Average rent by city:")
print(city_groups.sort_values(ascending=False))

print("\n" + "="*60 + "\n")

# Multiple aggregations at once
city_stats = df.groupby('city')['totalRent'].agg(['mean', 'median', 'std', 'count'])
print("Multiple statistics by city:")
print(city_stats.sort_values('mean', ascending=False))

In [ ]:
# Group by multiple columns
city_room_stats = df.groupby(['city', 'noRooms'])['totalRent'].mean().round(2)

print("Average rent by city and number of rooms:")
print(city_room_stats.head(15))

print("\n" + "="*60 + "\n")

# Unstack to create a pivot-like view
pivot_view = city_room_stats.unstack(fill_value=0)
print("Pivot view (cities × rooms):")
print(pivot_view)

In [ ]:
# Grouping with multiple columns and aggregations
grouped_multi = df.groupby('city').agg({
    'totalRent': ['mean', 'median', 'max', 'min'],
    'livingSpace': ['mean', 'median'],
    'noRooms': 'mean'
}).round(2)

print("Advanced grouping with different aggregations per column:")
print(grouped_multi)

**Visualizing Grouped Data**

Grouped data is perfect for comparative visualizations:

In [ ]:
# Bar chart: Average rent by city
plt.figure(figsize=(12, 5))
df_outlier_removed = df[df['totalRent'] < df['totalRent'].quantile(0.95)]

plt.subplot(1, 2, 1)
city_means = df_outlier_removed.groupby('city')['totalRent'].mean().sort_values(ascending=False)
city_means.plot(kind='bar', color='steelblue', edgecolor='black')
plt.xlabel('City', fontsize=12)
plt.ylabel('Average Rent (€)', fontsize=12)
plt.title('Average Rent by City', fontsize=14, fontweight='bold')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)

plt.subplot(1, 2, 2)
# Box plot: Rent distribution by city
cities = df_outlier_removed['city'].unique()
data_by_city = [df_outlier_removed[df_outlier_removed['city'] == city]['totalRent'].values for city in cities]
plt.boxplot(data_by_city, tick_labels=cities)
plt.xlabel('City', fontsize=12)
plt.ylabel('Total Rent (€)', fontsize=12)
plt.title('Rent Distribution by City', fontsize=14, fontweight='bold')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("Insights from grouping:")
print("1. Frankfurt has the highest average rent.")
print("2. Total rent variation within each city.")

**Practical Use Cases for Grouping** - How much does a balcony impact the total rent?

In [ ]:
# Example: Does heating type affect rent?
heating_impact = df.groupby('heatingType')['totalRent'].agg(['mean', 'count']).round(2)
heating_impact['mean_diff_from_overall'] = (heating_impact['mean'] - df['totalRent'].mean()).round(2)

print("Impact of heating type on rent:")
print(heating_impact.sort_values('mean', ascending=False))

print("\n" + "="*60 + "\n")

# Example: Apartments with balconies vs without
balcony_comparison = df.groupby('hasBalcony')['totalRent'].agg(['mean', 'median', 'count'])
balcony_comparison.index = ['No Balcony', 'Has Balcony']

print("Balcony impact on rent:")
print(balcony_comparison)
print(f"\nBalcony premium: {(balcony_comparison.loc['Has Balcony', 'mean'] - balcony_comparison.loc['No Balcony', 'mean']):.2f} €/month")

---

## 4. Data Manipulation and Cleaning

Real-world data is messy. This section covers essential cleaning techniques.

### Detecting and Removing Outliers

Outliers can severely impact model performance. They might represent:
- **Data entry errors** (e.g., rent = 10,000€ for a studio)
- **Measurement errors** (e.g., 1000 rooms)
- **Rare legitimate cases** (e.g., luxury penthouse)

**Common detection method:** Quantile-based filtering (1st-99th percentile)

In [ ]:
# Calculate quantile thresholds for totalRent
lower = df['totalRent'].quantile(0.01)
upper = df['totalRent'].quantile(0.99)

print(f"Outlier thresholds for totalRent:")
print(f"  Lower (1st percentile): {lower:.2f} €")
print(f"  Upper (99th percentile): {upper:.2f} €")

# Count outliers
outliers_count = ((df['totalRent'] < lower) | (df['totalRent'] > upper)).sum()
print(f"\nOutliers detected: {outliers_count} ({outliers_count/len(df)*100:.1f}%)")

# Remove outliers
df_clean = df[(df['totalRent'] >= lower) & (df['totalRent'] <= upper)].copy()
print(f"\nDataset size after outlier removal: {len(df_clean)} rows")

# Visualize before/after
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['totalRent'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_title('Before Outlier Removal', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Total Rent (€)')
axes[0].set_ylabel('Frequency')
axes[0].grid(alpha=0.3)

axes[1].hist(df_clean['totalRent'], bins=50, edgecolor='black', alpha=0.7, color='green')
axes[1].set_title('After Outlier Removal', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Total Rent (€)')
axes[1].set_ylabel('Frequency')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Understanding Missing Values

Missing values (`NaN` = Not a Number) are unavoidable in real datasets.

In [ ]:
# Visualize missing data pattern
plt.figure(figsize=(10, 6))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('Missing Value Pattern (Yellow = Missing)', fontsize=14, fontweight='bold')
plt.xlabel('Columns')
plt.tight_layout()
plt.show()

print(f"\nTotal missing values in dataset: {df.isnull().sum().sum()}")
print(f'\nMissing value counts: {df.isnull().sum()}')

#### Strategy 1: Dropping Missing Values

**Option A:** Drop rows with any missing values

In [ ]:
# Drop rows with ANY missing values
df_dropped = df.dropna()
print(f"Original: {len(df)} rows")
print(f"After dropping rows with any missing values: {len(df_dropped)} rows")
print(f"Lost: {len(df) - len(df_dropped)} rows ({(len(df) - len(df_dropped))/len(df)*100:.1f}%)")

# Drop rows with missing values in specific columns
df_dropped_subset = df.dropna(subset=['livingSpace', 'totalRent'])
print(f"\nAfter dropping rows with missing livingSpace or totalRent: {len(df_dropped_subset)} rows")

#### Strategy 2: Imputing Missing Values

Instead of dropping data, we can **fill** missing values with reasonable estimates.

**Common imputation strategies:**
- **Constant:** Fill with 0, -1, or "Unknown"
- **Mean/Median:** For numerical data
- **Mode:** For categorical data
- **Forward/Backward fill:** For time series
- **Calculated:** Infer from other columns

In [ ]:
# Example: Fill missing livingSpace with median
median_space = df['livingSpace'].median()
df_filled = df.copy()
df_filled['livingSpace'] = df_filled['livingSpace'].fillna(median_space)

print(f"Median living space: {median_space:.2f} m²")
print(f"Missing values before: {df['livingSpace'].isnull().sum()}")
print(f"Missing values after: {df_filled['livingSpace'].isnull().sum()}")

print("\n" + "="*60 + "\n")

# Example: Infer totalRent from baseRent + serviceCharge when totalRent is missing
# (We know: totalRent = baseRent + serviceCharge)
df_filled['totalRent_calculated'] = df_filled['baseRent'] + df_filled['serviceCharge'].fillna(0)
print("Calculated totalRent from baseRent + serviceCharge:")
print(df_filled[['baseRent', 'serviceCharge', 'totalRent', 'totalRent_calculated']].head())

#### Imputing Missing Values Using Feature Relationships

Sometimes when data is missing, we can **impute** (fill in) the missing values using relationships between features. Understanding how features relate to each other allows us to reconstruct missing information.

In our real estate dataset, `totalRent` has a mathematical relationship with two other features:
- `baseRent`: The base cost of renting the apartment
- `serviceCharge`: Additional charges for services (heating, water, maintenance, etc.)

**Mathematical relationship:** totalRent = baseRent + serviceCharge

This means if `totalRent` is missing but we have `baseRent` and `serviceCharge`, we can calculate the missing value instead of dropping the row or using simple imputation (mean/median).

In [ ]:
# Step 1: Check for actual missing totalRent values in the dataset
print("Step 1: Identify missing totalRent values")
print("="*60)

missing_total_rent = df['totalRent'].isnull().sum()
print(f"Missing totalRent values: {missing_total_rent}")

# Get indices of missing values
missing_mask = df['totalRent'].isnull()
missing_indices = df[missing_mask].index.tolist()

print(f"\nRows with missing totalRent (showing first 5):")
print(df.loc[missing_mask, ['baseRent', 'serviceCharge', 'totalRent']].head())

# Step 2: Impute missing values using the relationship
print("\n" + "="*60)
print("Step 2: Impute missing values using baseRent + serviceCharge")
print("="*60)

# Calculate missing totalRent values
df.loc[missing_mask, 'totalRent'] = df.loc[missing_mask, 'baseRent'] + df.loc[missing_mask, 'serviceCharge']

print(f"\nAfter imputation: {df['totalRent'].isnull().sum()} missing values")
print("\nImputed values (showing first 5):")
print(df.loc[missing_indices[:5], ['baseRent', 'serviceCharge', 'totalRent']])

print("\n" + "="*60)
print("✓ Missing values successfully imputed using feature relationship!")

# Step 3: Verify the relationship for all data
print("\n" + "="*60)
print("Step 3: Verify totalRent = baseRent + serviceCharge for all rows")
print("="*60)

df['calculated_totalRent'] = df['baseRent'] + df['serviceCharge']
matches = np.isclose(df['totalRent'], df['calculated_totalRent'], rtol=1e-5, equal_nan=True)
print(f"\n✓ {matches.sum()}/{len(df)} rows follow the relationship ({matches.mean() * 100:.1f}%)")

# Show some examples
print("\nExample rows:")
print(df[['baseRent', 'serviceCharge', 'totalRent', 'calculated_totalRent']].head(5))

# Clean up temporary column
df = df.drop('calculated_totalRent', axis=1)

### Feature Scaling: Standardization and Normalization

Machine learning algorithms often perform better when numerical features are on similar scales. Consider our dataset:
- `livingSpace`: ranges from ~20 to ~200 m²
- `yearBuilt`: ranges from ~1900 to ~2020
- `totalRent`: ranges from ~200 to ~3000 €

These different scales can cause problems:
1. **Gradient descent** converges slower with features on different scales
2. **Distance-based algorithms** (k-NN, k-means) are dominated by large-scale features
3. **Regularization** penalizes features unequally if scales differ

**Solution:** Scale features to comparable ranges using standardization or normalization.

#### Three Common Scaling Methods

We'll explore three fundamental scaling techniques:
1. **Standardization (Z-score normalization)** → Mean = 0, Std = 1
2. **Min-Max Scaling (0-1 normalization)** → Range = [0, 1]
3. **Robust Scaling** → Uses median and IQR (resistant to outliers)

Let's understand each method mathematically and practically.

#### Method 1: Standardization (Z-score Normalization)

**Standardization** transforms data to have **mean = 0** and **standard deviation = 1**. This is also called **Z-score normalization**.

For a feature $X$ with $n$ observations, the standardized value is:

$$z_i = \frac{x_i - \bar{x}}{\sigma_X}$$

Where:
- $x_i$ = original value
- $\bar{x}$ = mean of feature $X$
- $\sigma_X$ = standard deviation of $X$

**Key properties:**
- **Range:** Typically $[-3, +3]$ (most values within 3 standard deviations)
- **Distribution:** Preserves the shape of the original distribution
- **Outliers:** Retained (still visible in standardized data)
- **Units:** Dimensionless (values represent "number of standard deviations from mean")

**When to use:**
- Features follow approximately normal (Gaussian) distribution
- Algorithms assume normally distributed data (linear regression, logistic regression, LDA)
- You want to preserve outlier information
- Default choice for many ML algorithms

**Mathematical interpretation:**
- $z = 0$: Value is at the mean
- $z = +1$: Value is 1 standard deviation above the mean
- $z = -2$: Value is 2 standard deviations below the mean

In [ ]:
# Example: Standardization of livingSpace feature

# Extract the feature
living_space = df_clean['livingSpace'].dropna()

# Manual standardization to understand the process
mean_space = living_space.mean()
std_space = living_space.std()

# Apply standardization formula
living_space_standardized = (living_space - mean_space) / std_space

print("="*60)
print("STANDARDIZATION (Z-SCORE NORMALIZATION)")
print("="*60)
print(f"\nOriginal livingSpace:")
print(f"  Mean:     {mean_space:.2f} m²")
print(f"  Std Dev:  {std_space:.2f} m²")
print(f"  Min:      {living_space.min():.2f} m²")
print(f"  Max:      {living_space.max():.2f} m²")
print(f"  Range:    {living_space.max() - living_space.min():.2f} m²")

print(f"\nStandardized livingSpace:")
print(f"  Mean:     {living_space_standardized.mean():.10f}  (≈ 0)")
print(f"  Std Dev:  {living_space_standardized.std():.10f}  (≈ 1)")
print(f"  Min:      {living_space_standardized.min():.4f}")
print(f"  Max:      {living_space_standardized.max():.4f}")
print(f"  Range:    {living_space_standardized.max() - living_space_standardized.min():.4f}")

print("\n" + "="*60)
print("Example values:")
print(f"{'Original (m²)':<15} {'Standardized':<15} {'Interpretation'}")
print("-"*60)
sample_values = [30, mean_space - std_space, mean_space, mean_space + std_space, 100]
for val in sample_values:
    z_score = (val - mean_space) / std_space
    if abs(z_score) < 0.1:
        interp = "at the mean"
    elif z_score > 0:
        interp = f"{abs(z_score):.1f} std above mean"
    else:
        interp = f"{abs(z_score):.1f} std below mean"
    print(f"{val:>8.2f}        {z_score:>8.4f}        {interp}")
print("="*60)

#### Method 2: Min-Max Scaling (0-1 Normalization)

**Min-Max scaling** transforms data to a fixed range, typically **[0, 1]**. This is also called **0-1 normalization** or **min-max normalization**.

For a feature $X$ with $n$ observations, the scaled value is:

$$x_{\text{scaled}} = \frac{x_i - x_{\min}}{x_{\max} - x_{\min}}$$

To scale to an arbitrary range $[a, b]$:

$$x_{\text{scaled}} = a + \frac{(x_i - x_{\min})(b - a)}{x_{\max} - x_{\min}}$$

**Key properties:**
- **Range:** Exactly $[0, 1]$ for standard min-max (or $[a, b]$ for custom range)
- **Distribution:** Preserves the shape of the original distribution
- **Outliers:** Very sensitive! A single extreme value can compress all other values
- **Units:** Dimensionless (0 = minimum, 1 = maximum)

**When to use:**
- Features are bounded (have known min/max)
- You need features in a specific range (e.g., [0, 1] for neural networks)
- Distribution is not Gaussian
- Algorithms sensitive to magnitude (neural networks, k-NN)
- No significant outliers in the data

**Mathematical interpretation:**
- $x_{\text{scaled}} = 0$: Value is at the minimum
- $x_{\text{scaled}} = 0.5$: Value is at the midpoint
- $x_{\text{scaled}} = 1$: Value is at the maximum

**Warning:** New data might fall outside [0, 1] if it contains values beyond training min/max!

In [ ]:
# Example: Min-Max Scaling (0-1 normalization) of livingSpace feature

# Extract the feature (use same data for comparison)
living_space = df_clean['livingSpace'].dropna()

# Manual min-max scaling
min_space = living_space.min()
max_space = living_space.max()

# Apply min-max formula
living_space_minmax = (living_space - min_space) / (max_space - min_space)

print("="*60)
print("MIN-MAX SCALING (0-1 NORMALIZATION)")
print("="*60)
print(f"\nOriginal livingSpace:")
print(f"  Min:      {min_space:.2f} m²")
print(f"  Max:      {max_space:.2f} m²")
print(f"  Range:    {max_space - min_space:.2f} m²")
print(f"  Mean:     {living_space.mean():.2f} m²")

print(f"\nMin-Max Scaled livingSpace:")
print(f"  Min:      {living_space_minmax.min():.10f}  (exactly 0)")
print(f"  Max:      {living_space_minmax.max():.10f}  (exactly 1)")
print(f"  Range:    {living_space_minmax.max() - living_space_minmax.min():.10f}  (exactly 1)")
print(f"  Mean:     {living_space_minmax.mean():.4f}")

print("\n" + "="*60)
print("Example values:")
print(f"{'Original (m²)':<15} {'Min-Max [0,1]':<15} {'Interpretation'}")
print("-"*60)
sample_values = [min_space, 40, 60, living_space.mean(), 80, max_space]
for val in sample_values:
    scaled = (val - min_space) / (max_space - min_space)
    pct = scaled * 100
    print(f"{val:>8.2f}        {scaled:>8.4f}        {pct:.1f}% of range")
print("="*60)

# Demonstrate scaling to custom range [a, b]
a, b = -1, 1  # Scale to [-1, 1] instead of [0, 1]
living_space_custom = a + (living_space - min_space) * (b - a) / (max_space - min_space)

print(f"\nCustom range scaling to [{a}, {b}]:")
print(f"  Min:      {living_space_custom.min():.4f}")
print(f"  Max:      {living_space_custom.max():.4f}")
print(f"  Mean:     {living_space_custom.mean():.4f}")
print("="*60)

#### Visual Comparison

Let's visualize how each scaling method transforms the same data:

In [ ]:
# Compare all three scaling methods visually
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Original data
axes[0, 0].hist(living_space, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0, 0].set_xlabel('Living Space (m²)', fontsize=11)
axes[0, 0].set_ylabel('Frequency', fontsize=11)
axes[0, 0].set_title('Original Data', fontsize=13, fontweight='bold')
axes[0, 0].axvline(living_space.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean = {living_space.mean():.1f}')
axes[0, 0].axvline(living_space.median(), color='green', linestyle='--', linewidth=2, label=f'Median = {living_space.median():.1f}')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Standardized
axes[0, 1].hist(living_space_standardized, bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[0, 1].set_xlabel('Standardized Value', fontsize=11)
axes[0, 1].set_ylabel('Frequency', fontsize=11)
axes[0, 1].set_title('Standardization (μ=0, σ=1)', fontsize=13, fontweight='bold')
axes[0, 1].axvline(0, color='red', linestyle='--', linewidth=2, label='Mean = 0')
axes[0, 1].axvline(-1, color='gray', linestyle=':', alpha=0.7, label='±1 std')
axes[0, 1].axvline(1, color='gray', linestyle=':', alpha=0.7)
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# Min-Max scaled
axes[1, 0].hist(living_space_minmax, bins=50, edgecolor='black', alpha=0.7, color='green')
axes[1, 0].set_xlabel('Min-Max Scaled [0, 1]', fontsize=11)
axes[1, 0].set_ylabel('Frequency', fontsize=11)
axes[1, 0].set_title('Min-Max Scaling [0, 1]', fontsize=13, fontweight='bold')
axes[1, 0].axvline(0, color='blue', linestyle='--', linewidth=2, label='Min = 0')
axes[1, 0].axvline(1, color='purple', linestyle='--', linewidth=2, label='Max = 1')
axes[1, 0].axvline(0.5, color='gray', linestyle=':', alpha=0.7, label='Midpoint')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Key Observations:")
print("1. All methods preserve the distribution shape")
print("2. Standardization: data centered at 0, most values in [-3, 3]")
print("3. Min-Max: all values compressed to [0, 1]")

#### Scaling Multiple Features with scikit-learn

In practice, we use scikit-learn's scalers to transform entire datasets efficiently. Let's see how to apply scaling to multiple features at once:

**Important:** Always fit the scaler on **training data only**, then transform both training and test data. This prevents **data leakage** (where information from test set influences the model).

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.model_selection import train_test_split

# Prepare data: select numerical features
feature_cols = ['livingSpace', 'noRooms', 'yearBuilt']
df_scaling = df_clean[feature_cols].dropna()

# Split into train/test (80/20)
X_train, X_test = train_test_split(df_scaling, test_size=0.2, random_state=42)

print("="*70)
print("SCALING MULTIPLE FEATURES WITH SCIKIT-LEARN")
print("="*70)
print(f"\nOriginal data shapes:")
print(f"  Training: {X_train.shape}")
print(f"  Test:     {X_test.shape}")

# Show original data statistics
print("\n" + "-"*70)
print("Original Training Data Statistics:")
print(X_train.describe())

# Method 1: StandardScaler
print("\n" + "="*70)
print("1. STANDARDSCALER (Z-score normalization)")
print("="*70)
scaler_standard = StandardScaler()
X_train_std = scaler_standard.fit_transform(X_train)  # Fit on train, transform train
X_test_std = scaler_standard.transform(X_test)        # Transform test (no fitting!)

# Convert back to DataFrame for display
X_train_std_df = pd.DataFrame(X_train_std, columns=feature_cols)
print("\nStandardized Training Data:")
print(X_train_std_df.describe())
print(f"\nLearned parameters (from training data):")
print(f"  Means: {scaler_standard.mean_}")
print(f"  Std Devs: {scaler_standard.scale_}")

# Method 2: MinMaxScaler
print("\n" + "="*70)
print("2. MINMAXSCALER (0-1 normalization)")
print("="*70)
scaler_minmax = MinMaxScaler()
X_train_minmax = scaler_minmax.fit_transform(X_train)
X_test_minmax = scaler_minmax.transform(X_test)

X_train_minmax_df = pd.DataFrame(X_train_minmax, columns=feature_cols)
print("\nMin-Max Scaled Training Data:")
print(X_train_minmax_df.describe())
print(f"\nLearned parameters (from training data):")
print(f"  Mins: {scaler_minmax.data_min_}")
print(f"  Maxs: {scaler_minmax.data_max_}")


### Feature Engineering

**Feature engineering** is the process of creating new features from existing ones to provide additional insights or improve model performance. These derived features can capture relationships or ratios that aren't directly present in the raw data.

**Common feature engineering techniques:**
- **Ratios:** Price per unit, efficiency metrics (e.g., rent per square meter)
- **Combinations:** Totals, differences (e.g., age = current_year - birth_year)
- **Binning:** Converting continuous variables to categories
- **Polynomial features:** Squares, interactions between features

Let's create a new feature: **rent per square meter** - a key metric in real estate that normalizes rent by apartment size.

In [ ]:
# Create a new feature: rent per square meter
print("="*70)
print("FEATURE ENGINEERING: Creating rentPerSqm")
print("="*70)

# Calculate rent per square meter
df_clean['rentPerSqm'] = df_clean['totalRent'] / df_clean['livingSpace']

print("\nNew feature created: rentPerSqm = totalRent / livingSpace")
print("\n" + "-"*70)
print("Sample data with new feature:")
print(df_clean[['totalRent', 'livingSpace', 'rentPerSqm']].head(10))

print("\n" + "-"*70)
print("Statistics for rentPerSqm:")
print(df_clean['rentPerSqm'].describe())

print("\n" + "-"*70)
print("Comparison: Average rent per sqm by city")
rent_per_sqm_by_city = df_clean.groupby('city')['rentPerSqm'].agg(['mean', 'median', 'std']).round(2)
print(rent_per_sqm_by_city.sort_values('mean', ascending=False))

print("\n" + "="*70)
print("Why is this useful?")
print("="*70)
print("- Normalizes rent by apartment size (fair comparison)")
print("- Reveals geographic pricing differences more clearly")
print("- Helps identify if larger apartments have economies of scale")
print("- Can be a better predictor than absolute rent in some models")

# Visualize the new feature
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(df_clean['rentPerSqm'].dropna(), bins=50, edgecolor='black', alpha=0.7, color='teal')
plt.xlabel('Rent per Square Meter (€/m²)', fontsize=11)
plt.ylabel('Frequency', fontsize=11)
plt.title('Distribution of Rent per Square Meter', fontsize=13, fontweight='bold')
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
city_rent_per_sqm = df_clean.groupby('city')['rentPerSqm'].mean().sort_values(ascending=False)
city_rent_per_sqm.plot(kind='bar', color='teal', edgecolor='black')
plt.xlabel('City', fontsize=11)
plt.ylabel('Average Rent per m² (€)', fontsize=11)
plt.title('Average Rent per Square Meter by City', fontsize=13, fontweight='bold')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n✓ Feature 'rentPerSqm' added to df_clean (now {len(df_clean.columns)} columns)")

#### Encoding Categorical Variables

Machine learning models need numerical input. We must convert categorical variables to numbers.

**Common encoding strategies:**
1. **Label Encoding:** Map categories to integers (0, 1, 2, ...)
2. **One-Hot Encoding:** Create binary columns for each category
3. **Ordinal Encoding:** For ordered categories (low, medium, high → 0, 1, 2)

In [ ]:
# Example 1: Label Encoding
df_clean['city_encoded'] = df_clean['city'].astype('category').cat.codes
print("Label Encoding (city):")
print(df_clean[['city', 'city_encoded']].drop_duplicates().sort_values('city_encoded'))

print("\n" + "="*60 + "\n")

# Example 2: One-Hot Encoding using pd.get_dummies()
df_onehot = pd.get_dummies(df_clean['heatingType'], prefix='heating')
print("One-Hot Encoding (heatingType):")
print(df_onehot.head())

# Combine with original dataframe
df_encoded = pd.concat([df_clean, df_onehot], axis=1)
print(f"\nOriginal columns: {df_clean.shape[1]}")
print(f"After one-hot encoding: {df_encoded.shape[1]}")

---

## 5. Understanding Feature Relationships

Before building a model, we need to understand **which features are informative** and **how they relate** to our target variable.

### Foundational Concepts: Variance and Covariance

Before diving into correlation, we need to understand two fundamental statistical concepts that measure variability and relationships in data.

#### Variance

**Variance** measures how much a single variable spreads out from its mean. It quantifies the variability or dispersion of data points.

For a variable $X$ with $n$ observations, the **sample variance** is:

$$\text{Var}(X) = \sigma_X^2 = \frac{1}{n-1} \sum_{i=1}^{n}(x_i - \bar{x})^2$$

Where:
- $x_i$ = individual data points
- $\bar{x}$ = sample mean
- $n-1$ = degrees of freedom (Bessel's correction for sample variance)

**Key properties:**
- **Units:** Squared units of $X$ (e.g., if $X$ is in meters, variance is in m²)
- **Range:** $\text{Var}(X) \geq 0$ (always non-negative)
- **Interpretation:** Higher variance = more spread out data; lower variance = data clustered near mean
- **Special case:** $\text{Var}(X) = 0$ means all values are identical

**Standard deviation** ($\sigma_X$) is the square root of variance: $\sigma_X = \sqrt{\text{Var}(X)}$
- Same units as original data (easier to interpret)
- Most commonly reported measure of spread

#### Covariance

**Covariance** measures how two variables vary **together**. It indicates whether increases in one variable correspond to increases (positive covariance) or decreases (negative covariance) in another.

For two variables $X$ and $Y$ with $n$ observations, the **sample covariance** is:

$$\text{Cov}(X, Y) = \frac{1}{n-1} \sum_{i=1}^{n}(x_i - \bar{x})(y_i - \bar{y})$$

**Key properties:**
- **Units:** Product of units of $X$ and $Y$ (e.g., m × €)
- **Range:** $-\infty < \text{Cov}(X, Y) < +\infty$ (unbounded)
- **Interpretation:**
  - $\text{Cov}(X, Y) > 0$: Variables tend to increase together (positive relationship)
  - $\text{Cov}(X, Y) < 0$: When one increases, the other tends to decrease (negative relationship)
  - $\text{Cov}(X, Y) = 0$: No linear relationship
- **Special case:** $\text{Cov}(X, X) = \text{Var}(X)$ (covariance of a variable with itself is its variance)

**Problem with covariance:** The magnitude depends on the units and scale of the variables, making it hard to interpret. A covariance of 100 might be large or small depending on the data scale.

**Solution:** Standardize covariance → **Correlation coefficient**

### Correlation Analysis

**Correlation** quantifies the strength and direction of the linear relationship between two variables. The most commonly used measure is the **Pearson correlation coefficient** (denoted as $r$ or $\rho$).

The correlation coefficient is essentially a **normalized covariance** — it takes covariance and standardizes it to a fixed scale by dividing by the standard deviations of both variables.

#### Mathematical Definition

For two variables $X$ and $Y$ with $n$ observations, the Pearson correlation coefficient is defined as:

$$r_{X,Y} = \frac{\sum_{i=1}^{n}(x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_{i=1}^{n}(x_i - \bar{x})^2} \cdot \sqrt{\sum_{i=1}^{n}(y_i - \bar{y})^2}}$$

This can be equivalently written as:

$$r_{X,Y} = \frac{\text{Cov}(X, Y)}{\sigma_X \cdot \sigma_Y}$$

Where:
- $x_i, y_i$ = individual data points
- $\bar{x}, \bar{y}$ = sample means
- $\text{Cov}(X, Y)$ = covariance between $X$ and $Y$
- $\sigma_X, \sigma_Y$ = standard deviations of $X$ and $Y$

**Interpretation:**
- **Range:** $r \in [-1, +1]$
- **$r = +1$:** Perfect positive linear relationship (points lie exactly on a line with positive slope)
- **$r = -1$:** Perfect negative linear relationship (points lie exactly on a line with negative slope)
- **$r = 0$:** No linear relationship (but non-linear relationships may exist!)
- **$|r| \approx 0.7\text{-}1.0$:** Strong correlation
- **$|r| \approx 0.4\text{-}0.7$:** Moderate correlation
- **$|r| \approx 0.0\text{-}0.4$:** Weak correlation

**Important limitations:**
1. Measures **only linear** relationships (non-linear relationships may be missed)
2. Sensitive to **outliers** (a few extreme values can distort the coefficient)
3. **Correlation ≠ Causation** (correlated variables are not necessarily causally related)

#### Example: Computing Variance and Covariance

Let's calculate these measures for our real estate data using pandas:

In [ ]:
# Select two variables for analysis
living_space = df_clean['livingSpace'].dropna()
total_rent = df_clean.loc[living_space.index, 'totalRent']

# Calculate variance using pandas
var_space = living_space.var()  # Variance of living space
var_rent = total_rent.var()     # Variance of total rent

# Calculate standard deviations
std_space = living_space.std()  # σ_X
std_rent = total_rent.std()     # σ_Y

# Calculate covariance using pandas
cov_space_rent = living_space.cov(total_rent)

print("Variance and Covariance Analysis")
print("="*60)
print(f"\nLiving Space (X):")
print(f"  Mean:              {living_space.mean():.2f} m²")
print(f"  Variance:          {var_space:.2f} m²²")
print(f"  Standard Deviation: {std_space:.2f} m²")
print(f"  -> Data spreads ±{std_space:.2f} m² around the mean on average")

print(f"\nTotal Rent (Y):")
print(f"  Mean:              {total_rent.mean():.2f} €")
print(f"  Variance:          {var_rent:.2f} €²")
print(f"  Standard Deviation: {std_rent:.2f} €")
print(f"  -> Data spreads ±{std_rent:.2f} € around the mean on average")

print(f"\nCovariance between Living Space and Total Rent:")
print(f"  Cov(X, Y):         {cov_space_rent:.2f} m²·€")
print(f"  -> {'Positive' if cov_space_rent > 0 else 'Negative'} relationship: "
      f"larger apartments tend to have {'higher' if cov_space_rent > 0 else 'lower'} rent")

print(f"\nNote: Covariance magnitude is hard to interpret directly!")
print(f"      Is {cov_space_rent:.2f} a strong or weak relationship?")
print(f"      -> We need to normalize it -> Correlation coefficient")
print("="*60)

#### Computing Correlation: Step-by-Step Example

Let's manually compute the correlation between `livingSpace` and `totalRent` to understand how it works:

In [ ]:
# Computing correlation using pandas .corr() method
# Let's compute correlation between livingSpace and totalRent

# Extract variables (remove NaN values)
X_corr = df_clean['livingSpace'].dropna()
Y_corr = df_clean.loc[X_corr.index, 'totalRent']

# Calculate correlation using pandas built-in method
r_value = X_corr.corr(Y_corr)

print("Pearson Correlation Coefficient Calculation")
print("="*60)
print(f"Variables: livingSpace vs totalRent")
print(f"Sample size: n = {len(X_corr)}")
print("\nCalculating correlation:")
print(f"  r = {r_value:.6f}")
print("="*60)

print(f"\nInterpretation:")
print(f"  r = {r_value:.3f} indicates a {'strong' if abs(r_value) > 0.7 else 'moderate' if abs(r_value) > 0.4 else 'weak'} "
      f"{'positive' if r_value > 0 else 'negative'} linear relationship.")
print(f"  When living space increases, rent tends to {'increase' if r_value > 0 else 'decrease'}.")
print(f"\n  This means approximately {r_value**2:.1%} of the variance in rent can be explained by living space alone.")

#### Correlation Matrix

When working with multiple features, we compute pairwise correlations to form a **correlation matrix**. For $p$ features, this is a symmetric $p \times p$ matrix where element $(i,j)$ represents the correlation between features $i$ and $j$.

**Properties:**
- **Diagonal elements = 1:** Each feature is perfectly correlated with itself
- **Symmetric:** $r_{X,Y} = r_{Y,X}$
- **Bounded:** All off-diagonal elements are in $[-1, +1]$

In [ ]:
# Calculate correlation matrix for numerical features
numerical_cols = ['totalRent', 'baseRent', 'livingSpace', 'noRooms', 'yearBuilt', 'distanceFromCenter', 'rentPerSqm']
corr_matrix = df_clean[numerical_cols].corr()

print("Correlation Matrix:")
print(corr_matrix)

print("\n" + "="*60 + "\n")

# Correlation with target (totalRent)
target_corr = corr_matrix['totalRent'].sort_values(ascending=False)
print("Correlation with totalRent (sorted by strength):")
print(target_corr)

print("\n" + "="*60 + "\n")

# Identify strong correlations (excluding diagonal)
print("Identifying feature relationships:")
for i, col1 in enumerate(numerical_cols):
    for col2 in numerical_cols[i+1:]:
        r_value = corr_matrix.loc[col1, col2]
        if abs(r_value) > 0.7:
            relationship = "strong positive" if r_value > 0 else "strong negative"
            print(f"  {col1} - {col2}: r = {r_value:.3f} ({relationship})")
        elif abs(r_value) > 0.4:
            relationship = "moderate positive" if r_value > 0 else "moderate negative"
            print(f"  {col1} - {col2}: r = {r_value:.3f} ({relationship})")
        elif abs(r_value) > 0.2:
            relationship = "slightly positive" if r_value > 0 else "slightly negative"
            print(f"  {col1} - {col2}: r = {r_value:.3f} ({relationship})")

In [ ]:
# Visualize correlation matrix with heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8},
            fmt='.3f', vmin=-1, vmax=1)
plt.title('Correlation Heatmap (Pearson r)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("Reading the heatmap:")
print("- Color intensity indicates correlation strength")
print("- Red (positive): Variables increase together")
print("- Blue (negative): One increases as the other decreases")
print("- White (near zero): No linear relationship")
print("\nKey insights for modeling:")
print("- Strong correlations with totalRent -> potentially useful features")
print("- High correlations between features -> multicollinearity (redundant information)")
print("- baseRent and totalRent are highly correlated (expected, as totalRent = baseRent + serviceCharge)")

---

## 6. Connecting Pandas to Modeling

Now we understand how to prepare data with pandas. Let's see how this connects to the machine learning pipeline.

In [ ]:
embed_lecture_slides('03_NeuralNetworks/03-regression-deck.html#/machine-learning-pipeline-1/4')

### 6.1 From DataFrame to NumPy

Most scikit-learn models expect NumPy arrays as input. The conversion from pandas to NumPy is straightforward, but we need to be intentional about which columns to use as features and which is our target.

**Key concepts:**
- **Feature matrix (X):** Contains the input variables used for prediction
- **Target vector (y):** The variable we want to predict
- **Data types:** scikit-learn expects numerical arrays (floats or integers)

In [ ]:
# Start simple: Select ONE feature for our baseline model
# We'll use livingSpace since it had high correlation with totalRent
X_simple = df_clean[['livingSpace']].dropna()
y_simple = df_clean.loc[X_simple.index, 'totalRent']

# Convert to NumPy arrays
X = X_simple.values  # or .to_numpy()
y = y_simple.values

print(f"Feature matrix X:")
print(f"  Shape: {X.shape} → ({X.shape[0]} samples, {X.shape[1]} feature)")
print(f"  Type: {type(X)}")
print(f"\nTarget vector y:")
print(f"  Shape: {y.shape}")
print(f"  Type: {type(y)}")

print("\n" + "="*60 + "\n")
print("First 5 samples:")
print(f"X (livingSpace in m²):\n{X[:5].flatten()}")
print(f"\ny (totalRent in €):\n{y[:5]}")

### 6.2 Training a Simple Linear Model

Now that we have our data in NumPy format, we can train our first model. We'll start with a **simple linear regression** using just one feature: `livingSpace`.

**Linear regression** finds the best-fitting line: $\hat{y} = w \cdot x + b$

Where:
- $\hat{y}$ = predicted rent
- $x$ = living space (m²)
- $w$ = weight (slope) - how much rent increases per m²
- $b$ = bias (intercept) - base rent

The goal is to find $w$ and $b$ that minimize prediction errors.

In [ ]:
from sklearn.linear_model import LinearRegression

# Create and train the model
model = LinearRegression()
model.fit(X, y)

# Make predictions
y_pred = model.predict(X)

# Extract learned parameters
weight = model.coef_[0]
bias = model.intercept_

print("Model Training Complete!")
print("="*60)
print(f"Learned Parameters:")
print(f"  Weight (slope): {weight:.2f} €/m²")
print(f"  Bias (intercept): {bias:.2f} €")
print(f"\nModel equation:")
print(f"  totalRent = {weight:.2f} × livingSpace + {bias:.2f}")
print("="*60)

print(f"\nExample predictions:")
for sqm in [30, 50, 70, 100]:
    predicted_rent = weight * sqm + bias
    print(f"  {sqm:3d} m² → {predicted_rent:.2f} €")

In [ ]:
# Visualize the model fit
plt.figure(figsize=(10, 6))
plt.scatter(X, y, alpha=0.5, s=20, label='Actual data', color='steelblue')
plt.plot(X, y_pred, color='red', linewidth=2, label='Linear regression fit')
plt.xlabel('Living Space (m²)', fontsize=12)
plt.ylabel('Total Rent (€)', fontsize=12)
plt.title('Simple Linear Regression: Rent vs Living Space', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 6.3 Evaluating Model Performance

Training a model is only half the story. We need to **quantify how well it performs**. This helps us:
- Understand if our model is useful
- Compare different models
- Identify areas for improvement

For regression tasks, several metrics are commonly used. Let's explore them.

#### Mean Squared Error (MSE)

**MSE** measures the average squared difference between predictions and actual values:

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

**Characteristics:**
- **Units:** Squared units of the target (e.g., €²)
- **Interpretation:** Lower is better (0 = perfect)
- **Sensitivity:** Heavily penalizes large errors (due to squaring)
- **Problem:** Hard to interpret because of squared units

**Root Mean Squared Error (RMSE)** is often preferred: $\text{RMSE} = \sqrt{\text{MSE}}$
- Same units as the target (€), making it more interpretable
- "On average, predictions are off by X €"

#### Mean Absolute Error (MAE)

**MAE** measures the average absolute difference between predictions and actual values:

$$\text{MAE} = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$$

**Characteristics:**
- **Units:** Same as target (€)
- **Interpretation:** "On average, predictions are off by X €"
- **Sensitivity:** Treats all errors equally (no squaring)
- **Advantage:** More robust to outliers than MSE/RMSE
- **Disadvantage:** Less sensitive to large errors (which might be important!)

#### R² Score (Coefficient of Determination)

**R²** measures how much variance in the target is explained by the model:

$$R^2 = 1 - \frac{\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}{\sum_{i=1}^{n}(y_i - \bar{y})^2} = 1 - \frac{\text{MSE}_{\text{model}}}{\text{MSE}_{\text{baseline}}}$$

Where:
- $y_i$ = actual values
- $\hat{y}_i$ = predicted values  
- $\bar{y}$ = mean of actual values

**Characteristics:**
- **Range:** (-∞, 1], but typically [0, 1]
- **Interpretation:**
  - **R² = 1.0:** Perfect predictions (100% of variance explained)
  - **R² = 0.75:** Model explains 75% of variance
  - **R² = 0.0:** Model performs as well as just predicting the mean
  - **R² < 0.0:** Model performs worse than predicting the mean (bad!)
- **Advantage:** Scale-independent, easy to interpret as percentage
- **Usage:** Most common metric for regression tasks

**For the exercise:**
- Baseline (1 feature): R² > 0.75 expected
- Good models: R² > 0.85
- Excellent models: R² > 0.90

#### Computing Metrics for Our Model

Let's calculate all three metrics for our simple linear regression model:

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Calculate all metrics
mse = mean_squared_error(y, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y, y_pred)
r2 = r2_score(y, y_pred)

# Display results
print("="*60)
print("MODEL EVALUATION METRICS")
print("="*60)
print(f"\nMean Squared Error (MSE):       {mse:,.2f} €²")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f} €")
print(f"Mean Absolute Error (MAE):      {mae:.2f} €")
print(f"R² Score:                       {r2:.4f}")
print("="*60)

print(f"\nInterpretation:")
print(f"- Our model explains {r2*100:.2f}% of the variance in rent prices")
print(f"- On average, predictions are off by {mae:.2f} € (MAE)")

# Compare to baseline (predicting the mean)
baseline_pred = np.full_like(y, y.mean())
baseline_mae = mean_absolute_error(y, baseline_pred)
baseline_r2 = r2_score(y, baseline_pred)

print(f"\nComparison to baseline (predicting mean = {y.mean():.2f} €):")
print(f"  Baseline MAE:  {baseline_mae:.2f} €")
print(f"  Baseline R²:   {baseline_r2:.4f}")
print(f"  Improvement:   {((baseline_mae - mae) / baseline_mae * 100):.1f}% better (MAE)")

#### Visualizing Prediction Errors

Understanding **where** our model fails is as important as knowing **how much** it fails:

In [ ]:
# Calculate residuals (errors)
residuals = y - y_pred

# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Residual plot
axes[0].scatter(y_pred, residuals, alpha=0.5, s=20)
axes[0].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[0].set_xlabel('Predicted Rent (€)', fontsize=12)
axes[0].set_ylabel('Residual (Actual - Predicted) (€)', fontsize=12)
axes[0].set_title('Residual Plot', fontsize=14, fontweight='bold')
axes[0].grid(alpha=0.3)

# Plot 2: Actual vs Predicted
axes[1].scatter(y, y_pred, alpha=0.5, s=20)
axes[1].plot([y.min(), y.max()], [y.min(), y.max()], 'r--', linewidth=2, label='Perfect prediction')
axes[1].set_xlabel('Actual Rent (€)', fontsize=12)
axes[1].set_ylabel('Predicted Rent (€)', fontsize=12)
axes[1].set_title('Actual vs Predicted', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Residual Plot Interpretation:")
print("- Points around 0 = good predictions")
print("- Patterns in residuals suggest model is missing important information")
print("- Random scatter = model captures the relationship well")
print("\nActual vs Predicted Interpretation:")
print("- Points close to red line = accurate predictions")
print("- Spread from line = prediction errors")

### 6.4 Complete Workflow with Multiple Features

Now that we understand the basics, let's build a more sophisticated model using **multiple features** and proper **train/test splitting**.

In [ ]:
from sklearn.model_selection import train_test_split

# 1. Select multiple features
feature_cols = ['livingSpace', 'noRooms', 'yearBuilt', 'city_encoded']
target_col = 'totalRent'

df_clean = df_clean.dropna(subset=feature_cols + [target_col])
X_multi = df_clean[feature_cols].values
y_multi = df_clean[target_col].values

# 2. Split into training and validation sets
# This simulates real-world scenario: train on some data, evaluate on unseen data
X_train, X_val, y_train, y_val = train_test_split(
    X_multi, y_multi, test_size=0.2, random_state=42
)

print("Data Split:")
print(f"  Training set:   {X_train.shape[0]} samples ({(1-0.2)*100:.0f}%)")
print(f"  Validation set: {X_val.shape[0]} samples ({0.2*100:.0f}%)")

# 3. Train model on training data
model_multi = LinearRegression()
model_multi.fit(X_train, y_train)

# 4. Make predictions on both sets
y_train_pred = model_multi.predict(X_train)
y_val_pred = model_multi.predict(X_val)

# 5. Evaluate on both sets
train_r2 = r2_score(y_train, y_train_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
train_mae = mean_absolute_error(y_train, y_train_pred)

val_r2 = r2_score(y_val, y_val_pred)
val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
val_mae = mean_absolute_error(y_val, y_val_pred)

# Display results
print("\n" + "="*60)
print("MULTI-FEATURE MODEL PERFORMANCE")
print("="*60)
print(f"{'Metric':<20} {'Training':<15} {'Validation':<15}")
print("-"*60)
print(f"{'R² Score':<20} {train_r2:>8.4f}      {val_r2:>8.4f}")
print(f"{'RMSE (€)':<20} {train_rmse:>8.2f}      {val_rmse:>8.2f}")
print(f"{'MAE (€)':<20} {train_mae:>8.2f}      {val_mae:>8.2f}")
print("="*60)

print("\nFeature Importance (Coefficients):")
for feature, coef in zip(feature_cols, model_multi.coef_):
    print(f"  {feature:<15s}: {coef:>10.2f}")
print(f"  {'Intercept':<15s}: {model_multi.intercept_:>10.2f}")

# Compare to single-feature model
print(f"\n{'='*60}")
print("COMPARISON: Single vs Multi-Feature Model")
print(f"{'='*60}")
print(f"Single feature (livingSpace only):  R² = {r2:.4f}")
print(f"Multi-feature (4 features):         R² = {val_r2:.4f}")
print(f"Improvement:                        {(val_r2 - r2):.4f} ({((val_r2 - r2)/r2*100):.1f}%)")
print(f"{'='*60}")